# **COMP 2211 Introduction to Artificial Intelligence** #
## Lab 4 - K-Means Clustering with Pokémon ##

In this lab, you will cluster Pokémon based on their battle stats using **K-Means from scratch** with NumPy. The goal is to group similar Pokémon together and then interpret what each cluster represents.

We will explore groups with different HP, Attack, and Defense profiles. These three features alone do not establish a Pokémon's full battle role.

## Lab Tasks

We will use the supplied Pokémon stats CSV (800 rows before filtering). The goal is to cluster Pokémon into groups based on their battle statistics. No scikit-learn is allowed for clustering; we will implement K-Means ourselves with NumPy.

A quick rundown:
1. Load the dataset.
2. Filter to stronger Pokémon using `Total >= 500`.
3. Select a small set of battle features.
4. Standardize the data.
5. Implement K-Means from scratch.
6. Choose a reasonable K using the elbow method.
7. Cluster and visualize the results with PCA.

Dataset: `data/Pokemon.csv`

For the main lab, use these features:
- `HP`
- `Attack`
- `Defense`

Optional extension: try other combinations such as `Attack`, `Sp. Atk`, `Speed` and compare the clusters.

### Setup and data paths
Use Python 3 with NumPy, pandas, and Matplotlib. The optional PCA visualization also requires scikit-learn; scikit-learn is not allowed for task standardization or K-Means.

Locally, keep the supplied `data/` folder next to `files/`; the setup cell supports running from either the lab root or `files/`. In Colab, upload `Pokemon.csv` and `pokemon_cluster_subset.csv` to the working directory. Run cells from top to bottom after completing each TODO. Unfinished functions deliberately raise `NotImplementedError`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if __name__ == "__main__":
    # Local lab root, local files/ directory, or Colab upload directory.
    DATA_DIR = next((p for p in (Path("data"), Path("../data"), Path("."))
                     if (p / "Pokemon.csv").is_file()), None)
    if DATA_DIR is None:
        raise FileNotFoundError("Keep the supplied data/ folder, or upload Pokemon.csv to Colab.")


### Task 1: Load & Filter
Pokémon with a high `Total` stat are typically stronger overall, so we will focus on those first.

Your job:
1. Load the CSV file
2. Filter rows where `Total >= 500`
3. Select `features` in the requested order, then remove rows missing any selected feature (missing values in other columns do not remove a row).

Here are the types of the inputs and the types of the output to be returned by you:

**Inputs:**
1. file_name : str or pathlib.Path (a UTF-8 CSV path)
2. features : list[str] (an ordered list of requested feature names)

**Output:**

A pandas DataFrame containing only the selected feature columns for Pokémon with Total >= 500, with rows missing selected feature values removed.

In [ ]:
def load_n_filter(file_name, features):
    # TODO: Load the CSV file

    # TODO: Keep only Pokémon with Total >= 500

    # TODO: Select the chosen features and drop rows with missing values

    raise NotImplementedError("Task 1: load, filter, select features, and drop missing values")

if __name__ == "__main__":
    features = ['HP', 'Attack', 'Defense']
    df_filtered = load_n_filter(DATA_DIR / 'Pokemon.csv', features)
    X = df_filtered.to_numpy()

    print(f"Number of strong Pokémon (Total >= 500): {X.shape[0]}")
    print(f"Clustering on {X.shape[1]} battle features")

**Expected output:**

Number of strong Pokémon (Total >= 500): 247

Clustering on 3 battle features

### Task 2: Standardize the Features
**Why standardize?**

The stats are on different scales. If we do not standardize, one feature can dominate the distance calculation.

Your job: Use z-score standardization with mean = 0 and standard deviation = 1 for every nonconstant feature. Explicitly use the sample standard deviation (`np.std(X, axis=0, ddof=1)`); NumPy defaults to `ddof=0`.

Here is an explanation of the inputs and the output expected:

**Input:**

X : np.ndarray of shape (n_samples, n_features) containing the selected Pokémon features from Task 1

**Output:**

np.ndarray of shape (n_samples, n_features)

The standardized feature matrix where each nonconstant feature column has mean 0 and sample standard deviation 1 (`ddof=1`).

Use at least two finite samples. For a constant feature, replace its zero standard deviation with 1 so that the standardized column is all zeros (its standard deviation remains 0). Reject fewer than two samples with `ValueError`.


In [ ]:
# Note the following code is forbidden.
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_norm = scaler.fit_transform(X)
# print("No points for you!")

def standardize(X):
    # TODO: Standardize the features using NumPy only

    raise NotImplementedError("Task 2: implement sample z-score standardization")

if __name__ == "__main__":
    X_norm = standardize(X)
    print(X_norm[:5])

**Expected output (first five rows):**

```text
[[-0.25013515 -0.71874464 -0.39888320]
 [-0.25013515 -0.15306245  0.85740180]
 [-0.33465414 -0.65589106 -0.55591883]
 [-0.33465414  0.78974119  0.48051630]
 [-0.33465414 -0.02735530 -0.55591883]]
```


### Task 3: Implement K-Means from Scratch (NumPy Only)

Implement `initialize_centroids`, `assign_labels`, `update_centroids`, and `kmeans`.

- Inputs are finite numeric arrays of shape `(n_samples, n_features)` with at least one sample and feature; `1 <= k <= n_samples`, and `max_iter >= 1`.
- Initialize with `np.random.choice(n_samples, size=k, replace=False)`. Do not reset the random seed inside functions.
- Assign points by Euclidean distance. For ties, choose the smallest centroid index (`np.argmin`).
- Update nonempty clusters to their means. For an empty cluster `j`, use the data point `X[j % n_samples]` as a deterministic replacement; never divide by zero.
- Compute each centroid's movement with `np.linalg.norm(old - new, axis=1)`.
- If **all** movements are strictly less than `0.01`, stop and retain the **old centroids**, matching the original lab convention. Otherwise, accept the new centroids and continue, for at most `max_iter` iterations (default 75).
- Before returning, assign labels once more using the final centroids, including when the iteration limit is reached.

Return centroids of shape `(k, n_features)` and integer labels of shape `(n_samples,)`. Loops are allowed here; the no-loop restriction applies only to `compute_inertia`.


You may consider using `np.linalg.norm()` to calculate the Euclidean distance between two vectors/coordinates. It will make your task easier. You may read the documentation [here](https://numpy.org/doc/2.3/reference/generated/numpy.linalg.norm.html).

In [ ]:
def initialize_centroids(X, k):
    """Return k distinct sampled rows, shape (k, n_features)."""
    # TODO 1: use np.random.choice(..., replace=False).
    raise NotImplementedError("Task 3.1: initialize centroids")


def assign_labels(X, centroids):
    """Return nearest-centroid indices, shape (n_samples,); break ties by index."""
    # TODO 2: use broadcasting and np.linalg.norm(..., axis=2).
    raise NotImplementedError("Task 3.2: assign labels")


def update_centroids(X, labels, k):
    """Return cluster means, shape (k, n_features); handle empty clusters as specified."""
    # TODO 3
    raise NotImplementedError("Task 3.3: update centroids")


def kmeans(X, k, max_iter=75):
    """Return final centroids and their matching labels; follow the stopping contract."""
    # TODO 4: initialize, assign, update, check movement, and recompute final labels.
    raise NotImplementedError("Task 3.4: implement the K-Means loop")


In [ ]:
if __name__ == "__main__":
    np.random.seed(42)

    test_X = np.array([[1, 2, 3, 4], [2, 4, 6, 8], [4, 8, 10, 12]])
    test_k = 2

    print("centroids:\n", centroids := initialize_centroids(test_X, test_k))
    print("labels:\n", labels := assign_labels(test_X, centroids))
    print("updated centroids:\n", centroids := update_centroids(test_X, labels, test_k))

    centroids, labels = kmeans(X_norm, 4)
    print("final centroids:", centroids)
    print("final labels:", labels)

**Expected output:**

Your centroid and label values should match the logic of K-Means. The exact final clustering on the Pokémon data may vary slightly depending on random initialization.

### Task 4: The Elbow Method

Inertia (SSE or WCSS) is the **sum of squared Euclidean distances** from each point to its assigned centroid; it is not the average variance.

1. Run K-Means for each K from 2 to 18 (this exercise requires at least 18 samples).
2. Implement `compute_inertia(X, centroids, labels)` using NumPy operations, with no explicit loops or comprehensions. Return a scalar.
3. Return the 17 inertia values in increasing K order from `compute_all_inertias(X)`, and plot K against inertia.
4. Discuss a reasonable K at an apparent elbow. A clear elbow or a uniquely optimal K need not exist.

The loop over K is allowed. A single random run per K may produce an irregular curve because of local solutions; optional repeated initializations can help, but are not required by the graded function contract.


In [ ]:
def compute_inertia(X, centroids, labels):
    """
    Calculate inertia with ONLY NumPy operations and broadcasting
    return the inertia as a scalar value
    """
    # TODO
    # hint 1: use np.expand_dims() to help you get the right shape for broadcasting
    # hint 2: use np.arange() and labels to help you create the mask array
    # hint 3: multiply the distances array with the mask array to compute the final sum

    raise NotImplementedError("Task 4: compute inertia without loops")

def compute_all_inertias(X):
    inertias = []
    for k in range(2, 19):
        print(f"Running K-Means for K={k}...")
        # TODO: Run K-Means and append the inertia for each k
        raise NotImplementedError("Task 4: collect inertia values")

    return inertias

# Plot the elbow!
if __name__ == "__main__":
    np.random.seed(42)
    plt.figure(figsize=(10, 6))
    plt.plot(range(2, 19), compute_all_inertias(X_norm), marker='o', color='purple', linewidth=2.5)
    plt.xlabel('Number of Clusters (K)', fontsize=12)
    plt.ylabel('Inertia', fontsize=12)
    plt.title('Elbow Plot', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.xticks(range(2, 19))
    plt.show()

### Task 5: Run Final Clustering & Visualize the Clusters (Ungraded)

First, cluster the **same main dataset** used in Task 4 with your chosen K. There may be more than one reasonable choice of K; justify your choice using the elbow plot. Inspect the original feature means and the PCA projection; a 2D projection can hide or distort separation. Ellipses show two standard deviations of spread, not confidence intervals.

Optional extension: load `pokemon_cluster_subset.csv`, standardize it, and **recompute its own elbow plot** before choosing K. Do the same when changing features. The supplied subset is selected for illustration; do not assume its structure generalizes to the full dataset. HP/Attack/Defense can suggest defensive or offensive profiles, but do not determine a complete battle role.


In [ ]:
if __name__ == "__main__":
    # Use the main dataset and standardized features from Tasks 1-4.
    np.random.seed(42)
    optimal_K = None  # TODO: choose and justify an integer K from Task 4.
    if optimal_K is None:
        raise NotImplementedError("Choose optimal_K for the optional visualization")
    centroids, labels = kmeans(X_norm, optimal_K)

  # Reduce to 2D for better visualization using PCA
    from sklearn.decomposition import PCA  # only for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_norm)

    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        X_pca[:, 0], X_pca[:, 1], c=labels, cmap=plt.get_cmap('tab10', optimal_K),
        vmin=-0.5, vmax=optimal_K - 0.5, alpha=0.7, s=40, edgecolor='k', linewidth=0.2
    )

    from matplotlib.patches import Ellipse
    import matplotlib as mpl

    colors = scatter.cmap(np.arange(optimal_K))

    for k in range(optimal_K):
        cluster_points = X_pca[labels == k]
        if cluster_points.shape[0] < 2:
            continue

        mean = cluster_points.mean(axis=0)
        cov = np.cov(cluster_points, rowvar=False)

        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:, order]

        width, height = 2 * 2 * np.sqrt(np.maximum(vals, 0))
        angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))

        ell = Ellipse(
            xy=mean,
            width=width,
            height=height,
            angle=angle,
            facecolor='none',
            edgecolor=colors[k],
            linewidth=2,
            alpha=0.9,
        )
        plt.gca().add_patch(ell)

    plt.colorbar(scatter, label='Cluster', ticks=range(optimal_K))
    plt.title(f'Pokémon Clustered into {optimal_K} Groups', fontsize=14)
    plt.xlabel('PCA Dimension 1')
    plt.ylabel('PCA Dimension 2')
    plt.grid(True, alpha=0.3)
    plt.show()

    df_clusters = df_filtered.copy()
    df_clusters['cluster'] = labels
    print("Cluster characteristics (average feature values):")
    print(df_clusters.groupby('cluster')[features].mean().round(3))

## Grading Scheme (10 points total)

| Task | Function | Points | Criteria |
| --- | --- | ---: | --- |
| Task 1: Load and filter | `load_n_filter` | 2 | CSV loading, Total >= 500 filtering, requested feature order, and removal of missing selected values. |
| Task 2: Standardize | `standardize` | 1 | Sample z-scores (ddof=1), including constant-feature handling. |
| Task 3.1: Initialize centroids | `initialize_centroids` | 1 | Sample k distinct row indices using the specified random initialization. |
| Task 3.2: Assign labels | `assign_labels` | 1 | Nearest-centroid assignment with the specified tie rule. |
| Task 3.3: Update centroids | `update_centroids` | 1 | Cluster means and the specified empty-cluster handling. |
| Task 3.4: K-Means loop | `kmeans` | 1 | Iteration, stopping rule, and final centroid/label consistency. |
| Task 4.1: Inertia | `compute_inertia` | 2 | Correct sum of squared distances using NumPy without loops or comprehensions. |
| Task 4.2: Compare K values | `compute_all_inertias` | 1 | Return 17 inertia values for K=2 through 18 in order. |

Tasks 1–4 are worth **2 + 1 + 4 + 3 = 10 points**. Grading checks the functions above. The elbow plot and choice-of-K discussion support interpretation and carry no separate points. Task 5 (PCA visualization and exploration) is ungraded. Hidden tests may use different datasets; do not hard-code example results.

## Submission & Deadline

- **Deadline: Friday, 23 October 2026, 23:59.**
- Submit **`lab4_tasks.py`** directly to the [ZINC Online Submission System](https://zinc.cse.ust.hk/). Do not submit a ZIP archive, the notebook, or the reference solution.
- Complete `lab4_tasks.ipynb`, then export it in Colab via **File → Download → Download .py**. Ensure the exported file is named exactly `lab4_tasks.py`, and drag it into ZINC's submission panel.
- Remove or comment out debugging code. Keep demonstration data loading, printing, and plotting under `if __name__ == "__main__":` so importing your functions does not run them. Preserve the required function names and signatures.
- You may submit **up to 50 times** before the deadline. ZINC provides feedback after each submission. **Only the last submission is graded**, not the submission with the highest score.
- **Time limit: 300 seconds per test case.** Exceeding the limit results in a timeout and **0 points for that test case**. Use NumPy vectorization where appropriate; the no-loop restriction for `compute_inertia` still applies.
